# 1. 같은 사람 데이터로 과적합 줄이기

[명세](../../../ai-harness/ml-experiments/experiments/human-placement-candidate-early-stopping.md)의 첫 단계입니다. 원본 행동·배치·시간은 바꾸거나 증강하지 않습니다.

기존 공유 scorer는 실제 Full에서 Validation game-macro Top-1 43.56%였지만 BCE가 후반에 상승했습니다. 이번에는 **학습을 언제 끝낼지**만 바꿉니다. 기존30epoch report를 재사용하고, 아래 두 학습을 실제로 구현합니다.

```text
기존 Train149게임 → inner fit119 / monitor30
→ monitor BCE로 epoch 결정
→ 전체 Train149게임에서 새로 그 epoch만 학습
→ 외부 Validation31게임 평가
```

외부 Validation의 가장 좋은 epoch를 골라 다시 성적표를 내는 실험이 아닙니다. 새로운 모델 구조·L2 탐색·신규 수집·runtime은 범위 밖입니다.


## 2. 작은 손계산 예제

Patience=3, 개선 최소폭=0.0001입니다. Monitor BCE가 `0.60 → 0.50 → 0.50001 → 0.51 → 0.52`라면2epoch가 best이고3회 연속 개선이 없어5epoch에 중단합니다. 전체 Train에서는 **2epoch를 새로 학습**합니다. Inner 모델 weight는 가져오지 않습니다.

Inner monitor는 Train149게임 안에서만 고릅니다. 외부 Validation31게임은 이 선택 과정에 들어가지 않습니다. 내장 random-row early stopping은 같은 게임의 후보가 양쪽에 섞일 수 있어 사용하지 않습니다.


In [ ]:
import copy
import gc
import hashlib
import json
import math
import os
import platform
import tempfile
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-candidate-scoring"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_candidate_early_stopping.ipynb"
COMPARISON_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_candidate_model_comparison.ipynb"
COMPARISON_SHA = "282c64c675e59bf533a4b4c3c7e9b80ce3c04312b3e1c37f42724118203a04a3"
BASELINE_SHA = "16efde71ae0a783b9e99b9960c36736385d399920031b3431a8cbcc41f9731b7"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(COMPARISON_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == COMPARISON_SHA
base = {"__name__": "early_stopping_shared"}
for cell in selected:
    exec(compile(cell["source"], str(COMPARISON_NOTEBOOK), "exec"), base)
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "repo_input",
             "report_path", "publish_report", "reject_constant", "finite_float",
             "MLPClassifier", "StandardScaler", "threadpool_limits"):
    globals()[name] = base[name]
MAX_EPOCHS, PATIENCE, MIN_DELTA = 30, 3, 1e-4

def load_frozen_baseline():
    path = repo_input(os.environ.get("HUMAN_CANDIDATE_STOP_BASELINE",
        "data/ml-experiments/human-placement-candidate-scoring/model-comparison-run1.json"))
    require(sha256_file(path) == BASELINE_SHA, "FROZEN_BASELINE_CHANGED")
    report = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_constant,
                        parse_float=finite_float)
    base["check_model_report"](report)
    return report, path


In [ ]:
def split_inner_games(rows, monitor_count=30):
    require(rows and all(r["split"] == "train" for r in rows), "INNER_REQUIRES_TRAIN_ONLY")
    games = sorted({r["gameId"] for r in rows}, key=lambda g:
        (hashlib.sha256(("1729:early-stop:" + g).encode()).hexdigest(), g))
    require(type(monitor_count) is int and 0 < monitor_count < len(games), "INNER_GAME_COUNT_INVALID")
    return {"fitGameIds": sorted(games[monitor_count:]), "monitorGameIds": sorted(games[:monitor_count])}

def stopping_decision(losses):
    require(0 < len(losses) <= MAX_EPOCHS and np.isfinite(losses).all()
            and all(x >= 0 for x in losses), "INVALID_MONITOR_LOSS_HISTORY")
    best, best_epoch, waited = float("inf"), 0, 0
    for epoch, loss in enumerate(losses, 1):
        if loss < best - MIN_DELTA:
            best, best_epoch, waited = float(loss), epoch, 0
        else:
            waited += 1
        if waited >= PATIENCE:
            require(epoch == len(losses), "HISTORY_CONTINUES_AFTER_STOP")
            break
    stop = waited >= PATIENCE or len(losses) == MAX_EPOCHS
    return {"selectedEpoch": best_epoch, "selectedMonitorBce": best,
            "epochsRun": len(losses), "badEpochs": waited, "shouldStop": stop,
            "stopReason": "patience" if waited >= PATIENCE else "max_epochs" if stop else None}

def prepare_inner_arrays(rows, candidates):
    plan = split_inner_games(rows)
    fit_ids, monitor_ids = set(plan["fitGameIds"]), set(plan["monitorGameIds"])
    require(len(fit_ids) == 119 and len(monitor_ids) == 30 and not fit_ids & monitor_ids,
            "INNER_SPLIT_CHANGED")
    packs = []
    for ids in (fit_ids, monitor_ids):
        indices = [i for i, row in enumerate(rows) if row["gameId"] in ids]
        packs.append(base["pack_candidates"]([rows[i] for i in indices], [candidates[i] for i in indices]))
    fit, monitor = packs
    require(len(fit["rows"]) + len(monitor["rows"]) == 6417, "INNER_SEGMENT_COVERAGE_CHANGED")
    scaler = StandardScaler().fit(fit["x"], sample_weight=fit["scalerWeight"])
    require(np.isfinite(scaler.mean_).all() and np.isfinite(scaler.scale_).all()
            and (scaler.scale_ > 0).all(), "INNER_SCALER_INVALID")
    fit["x"] = scaler.transform(fit["x"], copy=False)
    monitor["x"] = scaler.transform(monitor["x"], copy=False)
    return fit, monitor, plan

def select_epoch_count(fit, monitor):
    require(all(r["split"] == "train" for r in fit["rows"] + monitor["rows"])
            and not {r["gameId"] for r in fit["rows"]} & {r["gameId"] for r in monitor["rows"]},
            "INNER_MONITOR_LEAKAGE")
    model = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", solver="adam",
        alpha=0.0001, batch_size=min(128, len(fit["y"])), learning_rate_init=0.001,
        shuffle=True, random_state=1729, early_stopping=False)
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.partial_fit(fit["x"], fit["y"], sample_weight=fit["weight"],
                              classes=np.array([0, 1]) if epoch == 1 else None)
        require(not caught, "INNER_FIT_WARNING: " + "; ".join(str(w.message) for w in caught))
        require(np.isfinite(model.loss_) and model.t_ == epoch * len(fit["y"]), "INNER_FIT_INVALID")
        values = {"epoch": epoch, "optimizationLoss": float(model.loss_)}
        for name, pack in (("fitBce", fit), ("monitorBce", monitor)):
            values[name] = base["binary_nll"](pack, base["predict_positive"](model, pack["x"]))
        history.append(values)
        decision = stopping_decision([r["monitorBce"] for r in history])
        print(f"inner epoch {epoch}: fit BCE={values['fitBce']:.6f}, monitor BCE={values['monitorBce']:.6f}")
        if decision["shouldStop"]:
            break
    require(all(np.isfinite(a).all() for a in (*model.coefs_, *model.intercepts_)), "INNER_MODEL_NONFINITE")
    return decision, history

def verify_refit_prefix(loss, history, reference):
    expected = reference["epochMetrics"]["shared_candidate_score"]
    require(0 < len(history) == len(loss) <= MAX_EPOCHS
            and np.allclose(loss, reference["optimizationLoss"]["shared_candidate_score"][:len(loss)],
                            atol=1e-12, rtol=0), "REFIT_LOSS_PREFIX_MISMATCH")
    for actual, frozen in zip(history, expected):
        require(actual["epoch"] == frozen["epoch"] and all(np.isclose(actual[s][m], frozen[s][m],
                atol=1e-12, rtol=0) for s in ("train", "validation")
                for m in ("top1", "top3", "mrr", "weightedBinaryNll")), "REFIT_METRIC_PREFIX_MISMATCH")


In [ ]:
def comparison_values(baseline_result, candidate_result, baseline_epoch, candidate_epoch):
    delta = {k: candidate_result["gameMacro"][k] - baseline_result["gameMacro"][k]
             for k in ("top1", "top3", "mrr")}
    bce_delta = candidate_epoch["validation"]["weightedBinaryNll"] - baseline_epoch["validation"]["weightedBinaryNll"]
    status = ("early_stopping_signal_observed" if bce_delta < -1e-12
              and delta["top1"] >= -1e-12 and delta["mrr"] >= -1e-12
              else "early_stopping_tradeoff_requires_review" if bce_delta < -1e-12 or delta["top1"] > 1e-12
              else "no_clear_early_stopping_gain")
    return {"gameMacroDelta": delta, "validationBceDelta": bce_delta,
            "baselineTop1Gap": baseline_epoch["train"]["top1"] - baseline_epoch["validation"]["top1"],
            "candidateTop1Gap": candidate_epoch["train"]["top1"] - candidate_epoch["validation"]["top1"],
            "analysisStatus": status}

def run_early_stopping(output_name):
    output = report_path(output_name)
    require((platform.python_version(), np.__version__, sklearn.__version__) ==
            ("3.12.13", "2.5.1", "1.9.0"), "PINNED_TRAINING_ENVIRONMENT_REQUIRED")
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    frozen, frozen_path = load_frozen_baseline()
    data, candidates, vocabulary, _, _, watched = base["read_model_inputs"]()
    watched[frozen_path] = BASELINE_SHA
    with threadpool_limits(limits=1):
        started = time.perf_counter()
        inner_fit, inner_monitor, plan = prepare_inner_arrays(data["train"], candidates["train"])
        decision, inner_history = select_epoch_count(inner_fit, inner_monitor)
        inner_counts = {name: {"games": len({r["gameId"] for r in pack["rows"]}),
                              "segments": len(pack["rows"]), "candidateRows": len(pack["y"])}
                        for name, pack in (("fit", inner_fit), ("monitor", inner_monitor))}
        inner_updates = math.ceil(len(inner_fit["y"]) / 128) * decision["epochsRun"]
        inner_presented = len(inner_fit["y"]) * decision["epochsRun"]
        print(f"Selected epoch={decision['selectedEpoch']}; inner elapsed={time.perf_counter()-started:.2f}s (non-canonical)")
        del inner_fit, inner_monitor
        gc.collect()
        train = base["pack_candidates"](data["train"], candidates["train"])
        validation = base["pack_candidates"](data["validation"], candidates["validation"])
        del candidates
        gc.collect()
        scaler = base["scale_candidate_inputs"](train, validation)
        require(np.allclose(scaler.mean_, frozen["training"]["sharedScalerMean"], atol=1e-12, rtol=0)
                and np.allclose(scaler.scale_, frozen["training"]["sharedScalerScale"], atol=1e-12, rtol=0),
                "FULL_TRAIN_SCALER_CHANGED")
        started = time.perf_counter()
        model, loss, history = base["fit_shared_scorer"](train, validation, decision["selectedEpoch"])
        verify_refit_prefix(loss, history, frozen)
        ranked = base["binary_rankings"](validation, base["predict_positive"](model, validation["x"]))
        print(f"Full-Train refit elapsed={time.perf_counter()-started:.2f}s (non-canonical)")
    for row, pred in zip(data["validation"], ranked):
        pred["baselineTargetOov"] = row["targetKey"] not in vocabulary
    result = base["topk_result"](ranked)
    baseline_result = frozen["results"]["shared_candidate_score"]
    comparison = comparison_values(baseline_result, result,
        frozen["epochMetrics"]["shared_candidate_score"][-1], history[-1])
    require(all(sha256_file(p) == digest for p, digest in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == source_sha
            and definition_sha(COMPARISON_NOTEBOOK) == COMPARISON_SHA, "SOURCE_OR_INPUT_CHANGED")
    report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "stage": "early_stopping",
        "runMode": "full", "fullCohortCovered": True, "policyDecisionEligible": True,
        "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
        "wholePolicyEvidence": False, "reachableCandidatesGenerated": False,
        "historicalRulesetVerified": False, "gameOverSafetyChecked": False, "testFileRead": False,
        "rawFileRead": False, "nodeInvoked": False, "dataAugmentation": False,
        "gameIdLeakage": 0, "modelFitCount": 2, "baselineFitCount": 0, "baselineReportReused": True,
        "smokeFitCount": 0, "outerValidationUsedForFit": False, "outerValidationUsedForEpochSelection": False,
        "outerValidationUsedForMonitoringDuringRefit": True, "sourceUnchanged": True,
        "newConditionRepeatDeterminismChecked": False, "refitPrefixReproduced": True,
        "nextDecision": "needs_policy_decision", "analysisStatus": comparison["analysisStatus"],
        "cohort": copy.deepcopy(frozen["cohort"]), "innerPlan": plan, "innerCounts": inner_counts,
        "stopping": {"maxEpochs": MAX_EPOCHS, "patience": PATIENCE, "minDelta": MIN_DELTA, **decision},
        "innerHistory": inner_history, "refitHistory": history, "refitOptimizationLoss": loss,
        "results": {"fixed_30": baseline_result, "early_stopped": result},
        "predictions": {"fixed_30": frozen["predictions"]["shared_candidate_score"], "early_stopped": ranked},
        "comparison": comparison,
        "byBaselineOov": [{"baselineTargetOov": flag,
            **base["topk_result"]([r for r in ranked if r["baselineTargetOov"] == flag])}
            for flag in (False, True)],
        "training": {"seed": 1729, "epochs": decision["selectedEpoch"], "hiddenSizes": [64],
            "alpha": .0001, "batchSize": 128, "optimizer": "adam", "learningRate": .001,
            "inputDimensions": 407, "parameterCount": sum(a.size for a in (*model.coefs_, *model.intercepts_)),
            "decisionWeightMin": float(np.add.reduceat(train["weight"], train["offsets"][:-1]).min()),
            "decisionWeightMax": float(np.add.reduceat(train["weight"], train["offsets"][:-1]).max()),
            "sampleWeightTotal": float(train["weight"].sum()), "scalerWeightTotal": float(train["scalerWeight"].sum()),
            "innerUpdates": inner_updates, "innerPresentedRows": inner_presented,
            "refitUpdates": math.ceil(len(train["y"])/128)*decision["selectedEpoch"],
            "refitPresentedRows": len(train["y"])*decision["selectedEpoch"],
            "baselineUpdatesReused": frozen["training"]["updates"]["shared"],
            "fitWarnings": 0, "convergenceAssessed": False, "blasThreads": 1,
            "fullScalerMean": scaler.mean_.tolist(), "fullScalerScale": scaler.scale_.tolist()},
        "provenance": {"sourceDefinitionSha256": source_sha, "comparisonDefinitionSha256": COMPARISON_SHA,
            "baselineReportSha256": BASELINE_SHA, "baselineProvenance": frozen["provenance"]},
        "environment": {"python": platform.python_version(), "numpy": np.__version__,
                        "sklearn": sklearn.__version__, "pandas": pd.__version__},
        "limits": ["inner_epoch_may_not_transfer_to_full_train", "bce_not_placement_top1_objective",
                   "development_validation_already_reused", "single_inner_split_single_seed",
                   "static_candidates_not_reachable", "no_artifact_or_runtime_promotion"]}
    check_early_report(report, frozen)
    publish_report(report, output)
    return report


In [ ]:
def check_early_report(report, frozen=None):
    if frozen is None:
        frozen, _ = load_frozen_baseline()
    require(report["reportSchemaVersion"] == 1 and report["experimentId"] == EXPERIMENT_ID
            and report["stage"] == "early_stopping" and report["runMode"] == "full"
            and report["fullCohortCovered"] is True and report["policyDecisionEligible"] is True
            and report["modelFitCount"] == 2 and report["baselineFitCount"] == 0
            and report["baselineReportReused"] is True and report["refitPrefixReproduced"] is True
            and report["gameIdLeakage"] == 0 and report["smokeFitCount"] == 0, "EARLY_REPORT_CONTRACT_INVALID")
    for key in ("canonicalResultEligible", "productionIntegration", "inferenceEligible", "wholePolicyEvidence",
                "reachableCandidatesGenerated", "historicalRulesetVerified", "gameOverSafetyChecked",
                "testFileRead", "rawFileRead", "nodeInvoked", "dataAugmentation",
                "outerValidationUsedForFit", "outerValidationUsedForEpochSelection"):
        require(report[key] is False, "EARLY_REPORT_FLAGS_INVALID")
    require(report["provenance"]["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and report["provenance"]["comparisonDefinitionSha256"] == COMPARISON_SHA
            and definition_sha(COMPARISON_NOTEBOOK) == COMPARISON_SHA
            and report["provenance"]["baselineReportSha256"] == BASELINE_SHA
            and report["provenance"]["baselineProvenance"] == frozen["provenance"]
            and report["cohort"] == frozen["cohort"], "EARLY_REPORT_PROVENANCE_INVALID")
    plan = report["innerPlan"]
    fit_ids, monitor_ids = set(plan["fitGameIds"]), set(plan["monitorGameIds"])
    require(len(fit_ids) == len(plan["fitGameIds"]) == 119 and
            len(monitor_ids) == len(plan["monitorGameIds"]) == 30 and not fit_ids & monitor_ids
            and not (fit_ids | monitor_ids) & {r["gameId"] for r in frozen["predictions"]["shared_candidate_score"]},
            "EARLY_REPORT_GAME_LEAKAGE")
    require(plan == split_inner_games([{"split": "train", "gameId": g} for g in fit_ids | monitor_ids]),
            "EARLY_REPORT_SPLIT_CHANGED")
    inner = report["innerHistory"]
    require([h["epoch"] for h in inner] == list(range(1, len(inner)+1)), "INNER_HISTORY_ORDER_INVALID")
    selected = stopping_decision([h["monitorBce"] for h in inner])
    require(report["stopping"] == {"maxEpochs": MAX_EPOCHS, "patience": PATIENCE,
            "minDelta": MIN_DELTA, **selected} and selected["shouldStop"], "STOPPING_DECISION_CHANGED")
    history, loss = report["refitHistory"], report["refitOptimizationLoss"]
    require(len(history) == len(loss) == selected["selectedEpoch"], "REFIT_EPOCH_COUNT_INVALID")
    verify_refit_prefix(loss, history, frozen)
    for name in ("fixed_30", "early_stopped"):
        rows = report["predictions"][name]
        require([r["segmentId"] for r in rows] ==
                [r["segmentId"] for r in frozen["predictions"]["shared_candidate_score"]]
                and all((a["gameId"], a["pieceType"], a["baselineTargetOov"]) ==
                        (b["gameId"], b["pieceType"], b["baselineTargetOov"])
                        for a, b in zip(rows, frozen["predictions"]["shared_candidate_score"]))
                and len(rows) == len({r["segmentId"] for r in rows}) == 1516
                and len({r["gameId"] for r in rows}) == 31
                and all(type(r["rank"]) is int and r["rank"] > 0 for r in rows)
                and report["results"][name] == base["topk_result"](rows), "EARLY_REPORT_AGGREGATION_INVALID")
    require(report["predictions"]["fixed_30"] == frozen["predictions"]["shared_candidate_score"]
            and report["results"]["fixed_30"] == frozen["results"]["shared_candidate_score"],
            "FROZEN_BASELINE_RESULT_CHANGED")
    values = comparison_values(report["results"]["fixed_30"], report["results"]["early_stopped"],
        frozen["epochMetrics"]["shared_candidate_score"][-1], history[-1])
    require(report["comparison"] == values and report["analysisStatus"] == values["analysisStatus"],
            "EARLY_REPORT_DECISION_INVALID")
    ranked = report["predictions"]["early_stopped"]
    require(sum(r["baselineTargetOov"] for r in ranked) == 152 and report["byBaselineOov"] ==
            [{"baselineTargetOov": flag, **base["topk_result"]([r for r in ranked if r["baselineTargetOov"] == flag])}
             for flag in (False, True)], "EARLY_REPORT_OOV_INVALID")
    t, counts = report["training"], report["innerCounts"]
    require(t["epochs"] == selected["selectedEpoch"] and t["fitWarnings"] == 0 and t["parameterCount"] == 26177
            and np.isclose(t["sampleWeightTotal"], 6417) and np.isclose(t["scalerWeightTotal"], 6417)
            and np.isclose(t["decisionWeightMin"], 1) and np.isclose(t["decisionWeightMax"], 1)
            and counts["fit"]["games"] == 119 and counts["monitor"]["games"] == 30
            and counts["fit"]["segments"] + counts["monitor"]["segments"] == 6417
            and counts["fit"]["candidateRows"] + counts["monitor"]["candidateRows"] == 147450
            and t["innerUpdates"] == math.ceil(counts["fit"]["candidateRows"]/128)*selected["epochsRun"]
            and t["refitUpdates"] == math.ceil(147450/128)*selected["selectedEpoch"], "EARLY_TRAINING_BUDGET_INVALID")
    json_bytes(report)
    return report

def show_early_report(report):
    frozen, _ = load_frozen_baseline()
    check_early_report(report, frozen)
    print(report["analysisStatus"], "| development evidence only; epoch selected without outer Validation")
    display(pd.DataFrame([{"strategy": name, **result["gameMacro"],
        "segments": result["segmentMicro"]["segments"]} for name, result in report["results"].items()]).set_index("strategy"))
    display(pd.DataFrame([{"scope": name, **values} for name, values in report["innerCounts"].items()]))
    display(pd.DataFrame(report["byBaselineOov"])[["baselineTargetOov", "gameMacro", "segmentMicro"]])
    games = pd.DataFrame([{"strategy": name, **r} for name, result in report["results"].items()
                         for r in result["perGame"]]).pivot(index="gameId", columns="strategy", values="top1")
    games["delta"] = games["early_stopped"] - games["fixed_30"]
    display(games)
    print("selected/refit epochs:", report["stopping"]["selectedEpoch"], "; inner ran:", report["stopping"]["epochsRun"])
    print("compute:", {k:v for k,v in report["training"].items() if "Updates" in k or "PresentedRows" in k})
    inner, refit = report["innerHistory"], report["refitHistory"]
    baseline = frozen["epochMetrics"]["shared_candidate_score"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for key in ("fitBce", "monitorBce"):
        axes[0,0].plot([h["epoch"] for h in inner], [h[key] for h in inner], label=key)
    axes[0,0].axvline(report["stopping"]["selectedEpoch"], color="grey", linestyle=":", label="selected epoch")
    axes[0,0].set(title="Inner Train only: decision-balanced BCE", xlabel="Epoch", ylabel="BCE")
    for split in ("train", "validation"):
        axes[0,1].plot([h["epoch"] for h in baseline], [h[split]["weightedBinaryNll"] for h in baseline],
                       label="fixed30 " + split, alpha=.6)
        axes[0,1].plot([h["epoch"] for h in refit], [h[split]["weightedBinaryNll"] for h in refit],
                       linestyle="--", label="refit " + split)
        axes[1,0].plot([h["epoch"] for h in baseline], [h[split]["top1"] for h in baseline], label=split)
    axes[0,1].set(title="Full Train / outer Validation BCE", xlabel="Epoch", ylabel="BCE")
    axes[1,0].axvline(report["stopping"]["selectedEpoch"], color="grey", linestyle=":", label="selected epoch")
    axes[1,0].set(title="Placement Top-1: segment micro", xlabel="Epoch", ylabel="Top-1", ylim=(0,1))
    axes[1,1].bar(range(len(games)), games["delta"])
    axes[1,1].axhline(0, color="black", linewidth=.8)
    axes[1,1].set(title="Outer Validation31 games: Top-1 delta", xlabel="Game index (ID-sorted)", ylabel="early - fixed30")
    for ax in (axes[0,0], axes[0,1], axes[1,0]):
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 3. 작은 내부 검사 — 성능 Smoke 학습 아님

Stop 규칙·게임 경계·가중치 전달·재현성 검사·출력 보호만 확인합니다. 가상 optimizer는 호출 계약을 검사하기 위한 것으로, 모델 성능으로 표시하거나 report로 저장하지 않습니다.


In [ ]:
def internal_checks():
    from unittest.mock import patch
    rows = [{"split": "train", "gameId": f"g{i//2}"} for i in range(10)]
    plan = split_inner_games(rows, 2)
    assert plan == split_inner_games(list(reversed(rows)), 2)
    assert len(plan["fitGameIds"]) == 3 and len(plan["monitorGameIds"]) == 2
    try:
        split_inner_games(rows + [{"split": "validation", "gameId": "outer"}], 2)
        raise AssertionError("outer validation accepted")
    except ValueError:
        pass
    d = stopping_decision([.6, .5, .50001, .51, .52])
    assert d["selectedEpoch"] == 2 and d["epochsRun"] == 5 and d["stopReason"] == "patience"
    assert stopping_decision([.5, .5, .5, .5])["selectedEpoch"] == 1
    assert stopping_decision([.5, .49995])["selectedEpoch"] == 1
    assert stopping_decision(list(np.linspace(1, .1, 30)))["stopReason"] == "max_epochs"
    for losses in ([], [float("nan")], [-.1], [.6, .5, .51, .52, .53, .54]):
        try:
            stopping_decision(losses)
            raise AssertionError("invalid stop history accepted")
        except ValueError:
            pass
    many_rows = [{"split": "train", "gameId": f"g{i%149}"} for i in range(6417)]
    expected_plan = split_inner_games(many_rows)
    expected_fit = set(expected_plan["fitGameIds"])
    def fake_pack(rows, candidates):
        assert len(rows) == len(candidates)
        return {"rows": rows, "x": np.array([[float(r["gameId"] in expected_fit)] for r in rows]),
                "scalerWeight": np.ones(len(rows))}
    class FakeScaler:
        def fit(self, x, sample_weight=None):
            assert (x == 1).all() and sample_weight.shape == (len(x),)
            self.mean_, self.scale_ = np.ones(1), np.ones(1)
            return self
        def transform(self, x, copy=False):
            return x
    with patch.dict(base, {"pack_candidates": fake_pack}), patch.dict(globals(), {"StandardScaler": FakeScaler}):
        inner_a, inner_b, actual_plan = prepare_inner_arrays(many_rows, [None]*6417)
    assert actual_plan == expected_plan and len(inner_a["rows"]) + len(inner_b["rows"]) == 6417
    assert (inner_b["x"] == 0).all()
    fit = {"rows": [{"split": "train", "gameId": "fit"}], "x": np.array([[0.], [1.]]),
           "y": np.array([1,0]), "weight": np.array([.5,.5])}
    monitor = {**fit, "rows": [{"split": "train", "gameId": "monitor"}], "x": np.array([[2.],[3.]])}
    class FakeMLP:
        def __init__(self, **kwargs):
            assert kwargs["early_stopping"] is False and kwargs["random_state"] == 1729
            self.epoch, self.t_, self.classes_ = 0, 0, np.array([0,1])
            self.coefs_, self.intercepts_ = [np.zeros((1,1))], [np.zeros(1)]
        def partial_fit(self, x, y, sample_weight=None, classes=None):
            assert x is fit["x"] and y is fit["y"] and np.array_equal(sample_weight, [.5,.5])
            assert classes is None or np.array_equal(classes, [0,1])
            self.epoch += 1; self.t_ += len(y); self.loss_ = .5
            return self
    def fake_positive(model, x):
        assert x is fit["x"] or x is monitor["x"]
        return np.full(2, model.epoch)
    def fake_loss(pack, p):
        return .2 if pack is fit else [.6,.5,.50001,.51,.52][int(p[0])-1]
    with patch.dict(globals(), {"MLPClassifier": FakeMLP}), patch.dict(base, {
            "predict_positive": fake_positive, "binary_nll": fake_loss}), patch("builtins.print"):
        decision, history = select_epoch_count(fit, monitor)
    assert decision == d and len(history) == 5
    try:
        select_epoch_count(fit, {**monitor, "rows": fit["rows"]})
        raise AssertionError("overlapping game accepted")
    except ValueError:
        pass
    example = {"epoch": 1, "train": {"top1": .5, "top3": .8, "mrr": .6, "weightedBinaryNll": .4},
               "validation": {"top1": .4, "top3": .7, "mrr": .5, "weightedBinaryNll": .5}}
    reference = {"epochMetrics": {"shared_candidate_score": [example]},
                 "optimizationLoss": {"shared_candidate_score": [.4]}}
    verify_refit_prefix([.4], [copy.deepcopy(example)], reference)
    changed = copy.deepcopy(example); changed["validation"]["top1"] += .1
    try:
        verify_refit_prefix([.4], [changed], reference)
        raise AssertionError("prefix drift accepted")
    except ValueError:
        pass
    a = {"gameMacro": {"top1": .4, "top3": .7, "mrr": .6}}
    b = {"gameMacro": {"top1": .5, "top3": .7, "mrr": .6}}
    c = copy.deepcopy(example); c["validation"]["weightedBinaryNll"] = .4
    assert comparison_values(a,b,example,c)["analysisStatus"] == "early_stopping_signal_observed"
    b["gameMacro"]["top1"] = .3
    assert comparison_values(a,b,example,c)["analysisStatus"] == "early_stopping_tradeoff_requires_review"
    with tempfile.TemporaryDirectory(dir="/private/tmp", prefix="early-stop-check-") as directory:
        root = Path(directory); keep = root / "keep.json"; keep.write_text("keep")
        link = root / "link.json"; link.symlink_to(keep)
        hard = root / "hard.json"; os.link(keep, hard)
        dangling = root / "dangling.json"; dangling.symlink_to(root / "absent.json")
        for path in (keep, link, hard, dangling, root / ".." / root.name / "alias.json", SOURCE_NOTEBOOK):
            try:
                report_path(str(path))
                raise AssertionError("unsafe output accepted")
            except ValueError:
                pass
        assert keep.read_text() == "keep"
    print("PASS: game split, patience/min_delta, Train-only scaler, fake optimizer weight/scope, prefix drift, decision, output protection; real model fit=0")

internal_checks()


## 4. Full — inner 선택 → 전체 Train 재학습 → 외부 평가

`tetris-ml` 커널에서 `RUN_FULL=True`와 아직 없는 `FULL_REPORT_PATH`를 지정합니다. 위 정의·내부 검사와 이 절의 설정→실행→결과를 실행하세요. Node 설정이나 기존 coverage/ID 모델 재실행은 필요 없습니다.

실제 신규 fit은2개입니다. Inner fit은 최대30epoch 중 patience로 종료하고, 두 번째 fit은 전체 Train149게임에서 선택된 epoch만 학습합니다. 기존30epoch 결과는 고정 report로 비교합니다. 재현 prefix가 다르면 중단합니다.

행렬만 최대 약565MiB이며 파싱/학습 메모리는 더 큽니다. Inner 배열은 refit 전에 해제합니다. 긴 Full을 자동 실행하지 않고, 성능 Smoke도 만들지 않습니다.


In [ ]:
RUN_FULL = False
FULL_REPORT_PATH = os.environ.get("HUMAN_CANDIDATE_STOP_REPORT")


In [ ]:
FULL_REPORT = None
if RUN_FULL:
    FULL_REPORT = run_early_stopping(FULL_REPORT_PATH)
    print(FULL_REPORT["analysisStatus"])
else:
    print("Full skipped. Actual inner selection and full-Train refit are implemented; no real fit or data load.")


In [ ]:
if FULL_REPORT is not None:
    show_early_report(FULL_REPORT)
else:
    print("No Full result yet. Internal fake checks are not performance evidence.")


## 5. 저장한 결과만 다시 보기

Full을 끄고 `REPORT_TO_READ`에 생성 JSON 경로를 지정합니다. 이 단계는 실제 모델을 학습하거나 dataset payload를 다시 파싱하지 않습니다.


In [ ]:
REPORT_TO_READ = None
SAVED_REPORT = None
if REPORT_TO_READ is not None:
    path = repo_input(REPORT_TO_READ)
    SAVED_REPORT = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_constant, parse_float=finite_float)
    show_early_report(SAVED_REPORT)
else:
    print("Set REPORT_TO_READ to inspect the saved early-stopping report.")


## 6. 해석·다음 단계

Inner BCE가 낮아지는 epoch를 선택했다고 외부 배치 Top-1까지 반드시 좋아지지는 않습니다. 전체149게임 재학습에서 학습 속도가 달라질 수 있습니다. 따라서 외부 BCE·게임별 배치 순위·Train/Validation gap을 함께 확인합니다.

기존30epoch 곡선의 좋은 시점을 사후에 골라 성능이라고 게시하지 않습니다. 외부 Validation은 이미 개발에 반복 사용했으므로 이번 결과도 독립 Test나 실제 gameplay 증거가 아닙니다.

결과를 해석하고 종료 정책을 판단한 뒤에만 **공유 scorer의 게임 수별 학습곡선**을 준비합니다. 이번에는 L2/모델 축소/새 데이터/증강/조작·리듬·Hold/runtime/commit을 시작하지 않습니다.
